# Выбор финального подхода до тестовой проверки

Цель — сравнить простое правило PAY_0 и логистическую регрессию v0
при двух лимитах ручной проверки: 10% и 20% клиентов.

Используется train.csv.

Тестовая выборка не используется на этапе выбора модели.

In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

TARGET = "default payment next month"
RANDOM_STATE = 42
N_SPLITS = 5
TOP_SHARES = [0.10, 0.20]

In [ ]:
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
train = pd.read_csv("train.csv")

assert train.shape == (24000, 25)
assert train[TARGET].sum() == 5309

train.shape

(24000, 25)

In [ ]:
payment_status_columns = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

train["TOTAL_BILL_AMT"] = train[bill_columns].sum(axis=1)
train["TOTAL_PAY_AMT"] = train[payment_columns].sum(axis=1)
train["NUM_DELINQ"] = (
    train[payment_status_columns] > 0
).sum(axis=1)

In [ ]:
feature_columns = [
    "LIMIT_BAL",
    "AGE",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    *payment_status_columns,
    *bill_columns,
    *payment_columns,
    "TOTAL_BILL_AMT",
    "TOTAL_PAY_AMT",
    "NUM_DELINQ"
]

X = train[feature_columns]
y = train[TARGET]

X.shape

(24000, 26)

In [ ]:
def evaluate_top_share(
    data,
    target_column,
    scores,
    share,
    method_name
):
    selected_count = int(np.ceil(len(data) * share))

    ranked = (
        data
        .assign(score=scores)
        .sort_values(
            ["score", "ID"],
            ascending=[False, True]
        )
        .head(selected_count)
    )

    y = ranked[target_column]
    base_rate = data[target_column].mean()

    return {
        "Метод": method_name,
        "Нагрузка, %": len(ranked) / len(data) * 100,
        "Событий в отборе": int(y.sum()),
        "Доля событий, %": y.mean() * 100,
        "Recall, %": y.sum() / data[target_column].sum() * 100,
        "Lift": y.mean() / base_rate
    }

In [ ]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numeric_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

model_v0 = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "logistic_regression",
            LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ]
)

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

for fold, (fit_index, valid_index) in enumerate(
    cv.split(X, y),
    start=1
):
    X_fit = X.iloc[fit_index]
    y_fit = y.iloc[fit_index]

    X_valid = X.iloc[valid_index]
    y_valid = y.iloc[valid_index]

    fold_model = model_v0
    fold_model.fit(X_fit, y_fit)

    fold_predictions = train.loc[
        X_valid.index,
        ["ID", TARGET, "PAY_0"]
    ].copy()

    fold_predictions["model_score"] = fold_model.predict_proba(
        X_valid
    )[:, 1]

    for share in TOP_SHARES:
        rule_result = evaluate_top_share(
            fold_predictions,
            TARGET,
            fold_predictions["PAY_0"].astype(float),
            share=share,
            method_name="Правило PAY_0"
        )

        model_result = evaluate_top_share(
            fold_predictions,
            TARGET,
            fold_predictions["model_score"],
            share=share,
            method_name="Логистическая регрессия v0"
        )

        rule_result["Фолд"] = fold
        rule_result["Нагрузка, %"] = share * 100

        model_result["Фолд"] = fold
        model_result["Нагрузка, %"] = share * 100

        results.append(rule_result)
        results.append(model_result)

results_df = pd.DataFrame(results)

results_df.round(2)

,Метод,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift,Фолд
0,Правило PAY_0,10.0,326,67.92,30.73,3.07,1
1,Логистическая регрессия v0,10.0,341,71.04,32.14,3.21,1
2,Правило PAY_0,20.0,515,53.65,48.54,2.43,1
3,Логистическая регрессия v0,20.0,546,56.88,51.46,2.57,1
4,Правило PAY_0,10.0,344,71.67,32.39,3.24,2
5,Логистическая регрессия v0,10.0,349,72.71,32.86,3.29,2
6,Правило PAY_0,20.0,515,53.65,48.49,2.42,2
7,Логистическая регрессия v0,20.0,540,56.25,50.85,2.54,2
8,Правило PAY_0,10.0,335,69.79,31.54,3.15,3
9,Логистическая регрессия v0,10.0,333,69.38,31.36,3.14,3


In [ ]:
summary = (
    results_df
    .groupby(["Нагрузка, %", "Метод"])
    .agg(
        Среднее_событий=("Событий в отборе", "mean"),
        Средняя_доля_событий=("Доля событий, %", "mean"),
        Средний_recall=("Recall, %", "mean"),
        Средний_lift=("Lift", "mean"),
        Минимальный_recall=("Recall, %", "min"),
        Максимальный_recall=("Recall, %", "max")
    )
    .reset_index()
)

summary.round(2)

,"Нагрузка, %",Метод,Среднее_событий,Средняя_доля_событий,Средний_recall,Средний_lift,Минимальный_recall,Максимальный_recall
0,10.0,Логистическая регрессия v0,338.4,70.50,31.87,3.19,31.36,32.86
1,10.0,Правило PAY_0,333.4,69.46,31.40,3.14,30.51,32.39
2,20.0,Логистическая регрессия v0,537.8,56.02,50.65,2.53,49.72,51.46
3,20.0,Правило PAY_0,508.4,52.96,47.88,2.39,46.80,48.54


## Результат выбора

Сравниваются правило PAY_0 и логистическая регрессия v0
на одинаковых пяти фолдах и при двух лимитах ручной проверки:
10% и 20% клиентов.

Для каждого лимита фиксируются число событий в отборе,
доля событий, recall и lift.

После просмотра результатов выбирается один подход для
финальной проверки на отдельной тестовой выборке.

Тестовая выборка не используется на этапе выбора.

## Протокол финальной проверки

До открытия test.csv зафиксированы следующие решения:

- выбранная модель: логистическая регрессия v0;
- признаки: 26 столбцов версии v0;
- гиперпараметры: C=1.0, max_iter=2000, solver="lbfgs";
- обучение: все 24 000 клиентов train.csv;
- проверка: все 6 000 клиентов test.csv;
- сценарии: top-10% и top-20%;
- сравнение: правило PAY_0 и модель v0;
- порядок при равных оценках: по возрастанию ID;
- метрики: число событий в отборе, доля событий, recall, lift;
- изменения модели, признаков и правил после просмотра теста
  не допускаются.